## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")
#groq_api_key



In [14]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.1', 'langchain': '1.3.14'}}, output_version=None, profile={'name': 'Llama 3.1 8B Instant', 'release_date': '2024-07-23', 'last_updated': '2024-07-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 131072, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001ABAA0DCE80>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001ABAA0DD3C0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [11]:
import requests
import os

api_key = os.environ.get("GROQ_API_KEY")
url = "https://api.groq.com/openai/v1/models"

headers = {
    "Authorization": f"Bearer {api_key}",
    "Content-Type": "application/json"
}

response = requests.get(url, headers=headers)

print(response.json())

{'object': 'list', 'data': [{'id': 'llama-3.3-70b-versatile', 'object': 'model', 'created': 1733447754, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 32768, 'hugging_face_id': 'meta-llama/Llama-3.3-70B-Instruct', 'name': 'Llama 3.3 70B', 'input_modalities': ['text'], 'output_modalities': ['text'], 'context_length': 131072, 'max_output_length': 32768, 'pricing': {'prompt': '0.00000059', 'completion': '0.00000079', 'image': '0', 'request': '0', 'input_cache_read': '0.000000295'}, 'supported_sampling_parameters': ['temperature', 'top_p', 'stop', 'seed', 'max_tokens'], 'supported_features': ['tools', 'json_mode']}, {'id': 'llama-3.1-8b-instant', 'object': 'model', 'created': 1693721698, 'owned_by': 'Meta', 'active': True, 'context_window': 131072, 'public_apps': None, 'max_completion_tokens': 131072, 'hugging_face_id': 'meta-llama/Llama-3.1-8B-Instruct', 'name': 'Llama 3.1 8B', 'input_modalities': ['text'], 'output_modalities': 

In [16]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Charan and I am a Chief AI Engineer")])

AIMessage(content="Nice to meet you, Charan. As a Chief AI Engineer, I'm sure you're well-versed in various areas of artificial intelligence and machine learning. What specific aspects of AI are you currently working on or interested in?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 49, 'total_tokens': 96, 'completion_time': 0.186769906, 'completion_tokens_details': None, 'prompt_time': 0.003659069, 'prompt_tokens_details': None, 'queue_time': 0.052220804, 'total_time': 0.190428975}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9b67-35fb-7ab2-bb08-578713b0a588-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 49, 'output_tokens': 47, 'total_tokens': 96})

In [18]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Charan and I am a Chief AI Engineer"),
        AIMessage(content="Nice to meet you, Charan. As a Chief AI Engineer, I'm sure you're well-versed in various areas of artificial intelligence and machine learning. What specific aspects of AI are you currently working on or interested in?\n"),
        HumanMessage(content="Hey What's my name and what do I do? and my age")
    ]
)

AIMessage(content="However, I don't have that information about you, Charan. \n\nYou told me earlier that you're a Chief AI Engineer, but you didn't mention your age. Also, you didn't provide any information about your name being different from Charan.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 119, 'total_tokens': 172, 'completion_time': 0.099675928, 'completion_tokens_details': None, 'prompt_time': 0.010949941, 'prompt_tokens_details': None, 'queue_time': 0.034753857, 'total_time': 0.110625869}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9b6a-dc7a-7e72-93dd-3ee40436d5ef-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 119, 'output_tokens': 53, 'total_tokens': 172})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [19]:
!pip install langchain_community

In [26]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

c:\Users\Charan\Desktop\NLP\Krish Naik\LANGCHAIN\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [27]:
with_message_history

RunnableWithMessageHistory(bound=RunnableBinding(bound=RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]), kwargs={}, config={}, config_factories=[], get_session_history=<function get_session_history at 0x000001ABAA0D77F0>, history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)])

In [28]:
config={"configurable":{"session_id":"chat1"}}

In [29]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Charan and I am a Chief AI Engineer")],
    config=config
)

In [30]:
response.content

"Nice to meet you, Charan. As a Chief AI Engineer, you must be involved in some exciting and innovative projects. What kind of AI-related work are you currently focusing on, and what are some of the most significant challenges you're facing in your field?"

In [31]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content="Your name is Charan, and you're a Chief AI Engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 117, 'total_tokens': 132, 'completion_time': 0.017277989, 'completion_tokens_details': None, 'prompt_time': 0.203440919, 'prompt_tokens_details': None, 'queue_time': 0.034342026, 'total_time': 0.220718908}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9b88-6e51-7c31-9f91-471e2870a752-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 117, 'output_tokens': 15, 'total_tokens': 132})

In [32]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"I don't have any information about your name. I'm a text-based AI assistant and our conversation just started, so I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you!"

In [33]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

"Hello John. It's nice to meet you. Is there anything I can help you with or would you like to chat about something in particular?"

In [35]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

In [36]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config
)
response.content

'Your name is Charan.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [37]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [38]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is Charan")]})

AIMessage(content="Nice to meet you, Charan. I'm happy to assist you with any questions or topics you'd like to discuss. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 34, 'prompt_tokens': 57, 'total_tokens': 91, 'completion_time': 0.057449509, 'completion_tokens_details': None, 'prompt_time': 0.022971815, 'prompt_tokens_details': None, 'queue_time': 0.035826848, 'total_time': 0.080421324}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_e2c608b1d6', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9b9b-1fcd-7bd3-9501-bc5e37021fff-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 34, 'total_tokens': 91})

In [39]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

c:\Users\Charan\Desktop\NLP\Krish Naik\LANGCHAIN\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [40]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is Charan")],
    config=config
)

response

AIMessage(content="Nice to meet you, Charan. I'm here to help with any questions or information you might need. How's your day going so far?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 57, 'total_tokens': 88, 'completion_time': 0.046325773, 'completion_tokens_details': None, 'prompt_time': 0.003646557, 'prompt_tokens_details': None, 'queue_time': 0.142091237, 'total_time': 0.04997233}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4848f70c04', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9b9f-445d-78c2-910c-bcc597691c52-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 57, 'output_tokens': 31, 'total_tokens': 88})

In [41]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Charan.'

In [42]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [44]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Charan")],"language":"Telugu"})
response.content

'హలో చరణ్! నేను మీ సహాయకుడిగా ఉంటాను. ఏ విషయాలపై మీతో మాట్లాడాలి?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [45]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

c:\Users\Charan\Desktop\NLP\Krish Naik\LANGCHAIN\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [46]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Charan")],"language":"Telugu"},
    config=config
)
repsonse.content

'హలో చరణ్ ! మీ సహాయం కోసం ఇక్కడున్నాను. ఏం తెలుసుకోవాలనుకుంటున్నారు?'

In [47]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "telugu"},
    config=config,
)

In [48]:
response.content

'మీ పేరు చరణ్.'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [49]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

c:\Users\Charan\Desktop\NLP\Krish Naik\LANGCHAIN\venv\lib\site-packages\langchain_core\language_models\base.py:448: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))
c:\Users\Charan\Desktop\NLP\Krish Naik\LANGCHAIN\venv\lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Charan\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate D

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [50]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

"Unfortunately, I don't have any information about your personal preferences. If you'd like to tell me, I'd be happy to chat about your favorite ice cream flavors!"

In [51]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked me a simple math problem: 2 + 2.'

In [52]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

c:\Users\Charan\Desktop\NLP\Krish Naik\LANGCHAIN\venv\lib\site-packages\IPython\core\interactiveshell.py:3579: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [53]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I don't think we've discussed your name yet. You haven't mentioned it, and I don't have any information about you. Would you like to tell me your name?"

In [54]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't ask a math problem. This conversation has just started, and I'm ready to help with any questions or problems you may have. What would you like to ask or discuss?"